# Feature Engineering & Technical Indicators for S&P 500

This notebook transforms the cleaned dataset into a rich, economically meaningful feature set for forecasting, volatility modeling, and regime detection.

In financial time series, **feature quality** often matters more than model complexity. The goal here is to engineer features that capture momentum, volatility dynamics, temporal dependence, seasonality, and market regimes — rather than adding indicators indiscriminately.

## Feature Categories

1. **Lag Features** — Short-term temporal dependencies
2. **Rolling Statistics** — Changing market conditions
3. **Technical Indicators** — Momentum and volatility measures
4. **Calendar Features** — Seasonal and temporal effects
5. **Regime Features** — Volatility states and market regimes

These features will support classical, GARCH, and machine learning models in later notebooks.

## Load Cleaned Dataset

In [ ]:

import pandas as pd
import numpy as np
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import Markdown, display
import os

pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark'
os.makedirs('../reports/figures', exist_ok=True)


Rows: 6,635
Date range: 2000-01-04 → 2026-05-22


In [9]:
# Load cleaned dataset from EDA
df = pd.read_parquet('../data/sp500_cleaned.parquet').copy()

# Preview dataset
df.head()

Price,Open,High,Low,Close,Volume,log_returns
Date,,,,,,
2000-01-04,1455.219971,1455.219971,1397.430054,1399.420044,1009000000,-0.039099
2000-01-05,1399.420044,1413.270020,1377.680054,1402.109985,1085500000,0.001920
2000-01-06,1402.109985,1411.900024,1392.099976,1403.449951,1092300000,0.000955
2000-01-07,1403.449951,1441.469971,1400.729980,1441.469971,1225200000,0.026730
2000-01-10,1441.469971,1464.359985,1441.469971,1457.599976,1064800000,0.011128


In [10]:

# Dataset validation summary function
def dataset_summary(df):
    return {
        'rows': len(df),
        'start': df.index.min().date(),
        'end': df.index.max().date()
    }

summary = dataset_summary(df)

print(f"Rows: {summary['rows']:,}")
print(f"Date range: {summary['start']} → {summary['end']}")

Rows: 6,635
Date range: 2000-01-04 → 2026-05-22


## Loaded Cleaned Dataset

The cleaned dataset produced during EDA is loaded as the foundation for feature engineering.

Using the EDA output ensures:

- Data integrity checks have already been completed  
- A provider-related zero-volume data anomaly has been identified and removed  
- Partial or incomplete latest observations have been excluded (T-1 enforcement)  
- Log returns have been computed and validated  
- Feature engineering begins from a consistent and reproducible baseline  

The dataset is stored in **Parquet** format to preserve structure and data types while improving loading efficiency.

In [11]:
display(Markdown(f"""
## Dataset Validation

The loaded dataset contains:

- **Rows:** {summary['rows']:,}
- **Temporal Coverage:** {summary['start']} → {summary['end']}

This validation confirms that feature engineering is being performed on the expected EDA output and that temporal coverage remains intact before constructing downstream model inputs.
"""))


## Dataset Validation

The loaded dataset contains:

- **Rows:** 6,635
- **Temporal Coverage:** 2000-01-04 → 2026-05-22

This validation confirms that feature engineering is being performed on the expected EDA output and that temporal coverage remains intact before constructing downstream model inputs.


# Lag Features

In [ ]:
lags = [1, 2, 3, 5]

for lag in lags:
    df[f'return_lag_{lag}'] = df['log_returns'].shift(lag)
    
df['close_lag_1'] = df['Close'].shift(1)
df['close_lag_2'] = df['Close'].shift(2)

# Additional Lagged features
df['range_lag_1'] = (df['High'] - df['Low']).shift(1)
df['volume_lag_1'] = df['Volume'].shift(1)

print(f"Created {len([col for col in df.columns if 'lag' in col])} Lag Features")

Created 14 Lag Features


### Lag Features — Short-Term Temporal Memory

After reviewing the PACF results from statistical diagnostics, the lag features were refined from 14 down to **8 high-value lags**:

**Selected Features:**
- `return_lag_1`, `return_lag_2`, `return_lag_3`, `return_lag_5`
- `close_lag_1`, `close_lag_2`
- `range_lag_1` — Previous day’s intraday range (High - Low)
- `volume_lag_1` — Previous day’s trading volume

**Rationale**

- PACF analysis showed the strongest direct dependence in the first 5 lags.
- Longer lags (10 and 20) added little unique information and increased multicollinearity risk.
- `range_lag_1` captures recent volatility intensity.
- `volume_lag_1` adds liquidity context, which often signals regime shifts.

These features provide the model with structured access to recent market behavior (direction, level, volatility, and participation) without excessive redundancy.

# Rolling Statistics 

In [ ]:
windows = [5, 10, 21, 30, 60]

for w in windows:
    # Volatility measures
    df[f'vol_roll_{w}'] = df['log_returns'].rolling(window=w).std()

    # Mean dynamics
    df[f'return_mean_roll_{w}'] = df['log_returns'].rolling(window=w).mean()

    # Momentum (price relative to moving average)
    df[f'momentum_{w}'] = df['Close'] / df['Close'].rolling(window=w).mean() - 1


print(f'Created rolling features across {len(windows)} windows (5 to 60 days)')

### Rolling Statistics

We created rolling features using windows of 5, 10, 21, 30, and 60 days.

Although the dataset contains only trading days, these window sizes follow common conventions in quantitative finance and risk management, blending trading-time and calendar-time perspectives.

**Window Selection Rationale:**

- **5 & 10 days**: Capture short-term behavior and immediate market reactions.
- **21 days**: Represents a standard trading month, widely used in technical analysis.
- **30 days**: Industry benchmark for volatility measurement and risk reporting.
- **60 days**: Provides a medium-term view (roughly three calendar months) to identify sustained regimes.

**Features Created:**

- `vol_roll_*` — Rolling volatility (standard deviation of log returns)
- `return_mean_roll_*` — Rolling average of returns
- `momentum_*` — Price relative to its moving average

These features are particularly important because they directly address the **volatility clustering** and **conditional heteroskedasticity** identified during statistical diagnostics. They allow models to observe how risk and market behavior evolve across multiple time horizons, rather than relying on single-period observations.